In [1]:
import sys

sys.path.append("../src")

In [2]:
import pandas as pd

from smogcast.ingest.gios import get_all_stations

In [3]:
selected_stations_df = pd.read_parquet(
    "../data/raw/gios_capital_stations_poland.parquet"
)

In [4]:
selected_stations_df.head()

,station_id,station_name,city,voivodeship,pm10_count,pm25_count
0,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,1,1
1,156,"Bydgoszcz, Pl. Poznański",Bydgoszcz,KUJAWSKO-POMORSKIE,1,1
2,258,"Lublin, ul. Śliwińskiego",Lublin,LUBELSKIE,1,1
3,361,Gorzów Wlkp. ul. Kosynierów Gdyńskich,Gorzów Wielkopolski,LUBUSKIE,2,1
4,295,"Łódź, ul. Czernika",Łódź,ŁÓDZKIE,1,2


In [5]:
stations = get_all_stations()

In [6]:
coordinates_df = pd.DataFrame(
    [
        {
            "station_id": station["Identyfikator stacji"],
            "latitude": float(station["WGS84 φ N"]),
            "longitude": float(station["WGS84 λ E"]),
        }
        for station in stations
    ]
)

In [7]:
weather_stations_df = selected_stations_df.merge(
    coordinates_df,
    on="station_id",
    how="left",
)

In [8]:
weather_stations_df[
    [
        "station_id",
        "station_name",
        "city",
        "latitude",
        "longitude",
    ]
]

,station_id,station_name,city,latitude,longitude
0,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,51.129378,17.029250
1,156,"Bydgoszcz, Pl. Poznański",Bydgoszcz,53.121764,17.987906
2,258,"Lublin, ul. Śliwińskiego",Lublin,51.273078,22.551675
3,361,Gorzów Wlkp. ul. Kosynierów Gdyńskich,Gorzów Wielkopolski,52.738214,15.228667
4,295,"Łódź, ul. Czernika",Łódź,51.758050,19.529786
5,400,"Kraków, Aleja Krasińskiego",Kraków,50.057678,19.926189
6,530,"Warszawa, Al. Niepodległości",Warszawa,52.219298,21.004724
7,590,"Opole, Os. Armii Krajowej",Opole,50.676856,17.950278
8,671,"Rzeszów, Al. Rejtana",Rzeszów,50.024242,22.010575
9,609,"Białystok, ul. Warszawska",Białystok,53.129306,23.181744


In [9]:
weather_stations_df[
    ["latitude", "longitude"]
].isna().sum()

latitude     0
longitude    0
dtype: int64

In [10]:
import pandas as pd

from smogcast.ingest.weather import get_weather_for_location

In [11]:
station = weather_stations_df.iloc[0]

In [12]:
station[
    [
        "station_id",
        "station_name",
        "city",
        "latitude",
        "longitude",
    ]
]

station_id                                        117
station_name    Wrocław, wyb. Conrada-Korzeniowskiego
city                                          Wrocław
latitude                                    51.129378
longitude                                    17.02925
Name: 0, dtype: object

In [13]:
weather_test = get_weather_for_location(
    station["station_id"],
    station["station_name"],
    station["city"],
    station["voivodeship"],
    station["latitude"],
    station["longitude"],
    "2026-08-01",
    "2026-08-03",
)

In [14]:
type(weather_test)

list

In [15]:
len(weather_test)

72

In [16]:
weather_test_df = pd.DataFrame(weather_test)

In [17]:
print(weather_test_df["timestamp_utc"].min())
print(weather_test_df["timestamp_utc"].max())

2026-08-01T00:00
2026-08-03T23:00


In [18]:
from smogcast.ingest.weather import get_weather_for_stations

In [19]:
station_records = weather_stations_df.to_dict("records")

In [20]:
len(station_records)

16

In [21]:
start_date = "2025-08-20"
end_date = "2026-08-20"

In [22]:
all_weather, failed_stations = get_weather_for_stations(
    station_records,
    start_date,
    end_date,
)

1/16 | DOLNOŚLĄSKIE | Wrocław | stacja 117
Timeout dla stacji 117. Próba 1/3
2/16 | KUJAWSKO-POMORSKIE | Bydgoszcz | stacja 156
3/16 | LUBELSKIE | Lublin | stacja 258
4/16 | LUBUSKIE | Gorzów Wielkopolski | stacja 361
5/16 | ŁÓDZKIE | Łódź | stacja 295
6/16 | MAŁOPOLSKIE | Kraków | stacja 400
Timeout dla stacji 400. Próba 1/3
7/16 | MAZOWIECKIE | Warszawa | stacja 530
8/16 | OPOLSKIE | Opole | stacja 590
9/16 | PODKARPACKIE | Rzeszów | stacja 671
Timeout dla stacji 671. Próba 1/3
10/16 | PODLASKIE | Białystok | stacja 609
Timeout dla stacji 609. Próba 1/3
11/16 | POMORSKIE | Gdańsk | stacja 729
12/16 | ŚLĄSKIE | Katowice | stacja 814
Timeout dla stacji 814. Próba 1/3
13/16 | ŚWIĘTOKRZYSKIE | Kielce | stacja 11195
14/16 | WARMIŃSKO-MAZURSKIE | Elbląg | stacja 861
Timeout dla stacji 861. Próba 1/3
15/16 | WIELKOPOLSKIE | Poznań | stacja 944
16/16 | ZACHODNIOPOMORSKIE | Szczecin | stacja 986
Timeout dla stacji 986. Próba 1/3


In [23]:
len(all_weather)

140544

In [24]:
len(failed_stations)

0

In [25]:
weather_df = pd.DataFrame(all_weather)

In [26]:
weather_df.head()

,station_id,station_name,city,voivodeship,latitude,longitude,timestamp_utc,temperature_2m,relative_humidity_2m,wind_speed_10m
0,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,51.129378,17.02925,2025-08-20T00:00,15.5,78,4.8
1,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,51.129378,17.02925,2025-08-20T01:00,13.8,86,5.0
2,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,51.129378,17.02925,2025-08-20T02:00,13.1,89,4.3
3,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,51.129378,17.02925,2025-08-20T03:00,12.6,91,3.2
4,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,51.129378,17.02925,2025-08-20T04:00,12.4,92,2.3


In [27]:
weather_df["timestamp_utc"] = pd.to_datetime(
    weather_df["timestamp_utc"],
    utc=True,
)

In [28]:
print("Liczba pomiarów:", len(weather_df))
print("Liczba stacji:", weather_df["station_id"].nunique())
print("Najstarszy pomiar:", weather_df["timestamp_utc"].min())
print("Najnowszy pomiar:", weather_df["timestamp_utc"].max())
print("Nieudane stacje:", len(failed_stations))

Liczba pomiarów: 140544
Liczba stacji: 16
Najstarszy pomiar: 2025-08-20 00:00:00+00:00
Najnowszy pomiar: 2026-08-20 23:00:00+00:00
Nieudane stacje: 0


In [29]:
weather_df[
    [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
    ]
].isna().sum()

temperature_2m          0
relative_humidity_2m    0
wind_speed_10m          0
dtype: int64

In [30]:
weather_df.duplicated(
    subset=["station_id", "timestamp_utc"]
).sum()

np.int64(0)

In [31]:
weather_df.to_parquet(
    "../data/raw/open_meteo_weather_1y.parquet",
    index=False,
)

In [32]:
failed_stations_df = pd.DataFrame(
    failed_stations,
    columns=["station_id"],
)

In [33]:
failed_stations_df.to_csv(
    "../data/raw/open_meteo_failed_stations_1y.csv",
    index=False,
)

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))

from smogcast.ingest.weather import get_weather_for_stations

In [2]:
CURRENT_WEATHER_PATH = "../data/raw/open_meteo_weather_1y.parquet"
OLDER_WEATHER_PATH = "../data/raw/open_meteo_weather_older.parquet"
FULL_WEATHER_PATH = "../data/raw/open_meteo_weather_3y.parquet"

START_DATE = "2023-08-20"
END_DATE = "2025-08-19"

In [3]:
current_weather = pd.read_parquet(CURRENT_WEATHER_PATH)

current_weather.head()

,station_id,station_name,city,voivodeship,latitude,longitude,timestamp_utc,temperature_2m,relative_humidity_2m,wind_speed_10m
0,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,51.129378,17.02925,2025-08-20 00:00:00+00:00,15.5,78,4.8
1,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,51.129378,17.02925,2025-08-20 01:00:00+00:00,13.8,86,5.0
2,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,51.129378,17.02925,2025-08-20 02:00:00+00:00,13.1,89,4.3
3,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,51.129378,17.02925,2025-08-20 03:00:00+00:00,12.6,91,3.2
4,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,51.129378,17.02925,2025-08-20 04:00:00+00:00,12.4,92,2.3


In [4]:
station_records_df = (
    current_weather[
        [
            "station_id",
            "station_name",
            "city",
            "voivodeship",
            "latitude",
            "longitude",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

station_records = station_records_df.to_dict(
    orient="records"
)

print("Liczba stacji:", len(station_records))

Liczba stacji: 16


In [5]:
older_weather_records, failed_stations = get_weather_for_stations(
    station_records=station_records,
    start_date=START_DATE,
    end_date=END_DATE,
)

1/16 | DOLNOŚLĄSKIE | Wrocław | stacja 117
2/16 | KUJAWSKO-POMORSKIE | Bydgoszcz | stacja 156
3/16 | LUBELSKIE | Lublin | stacja 258
4/16 | LUBUSKIE | Gorzów Wielkopolski | stacja 361
5/16 | ŁÓDZKIE | Łódź | stacja 295
6/16 | MAŁOPOLSKIE | Kraków | stacja 400
7/16 | MAZOWIECKIE | Warszawa | stacja 530
8/16 | OPOLSKIE | Opole | stacja 590
9/16 | PODKARPACKIE | Rzeszów | stacja 671
Timeout dla stacji 671. Próba 1/3
10/16 | PODLASKIE | Białystok | stacja 609
Timeout dla stacji 609. Próba 1/3
Timeout dla stacji 609. Próba 2/3
11/16 | POMORSKIE | Gdańsk | stacja 729
Timeout dla stacji 729. Próba 1/3
12/16 | ŚLĄSKIE | Katowice | stacja 814
13/16 | ŚWIĘTOKRZYSKIE | Kielce | stacja 11195
Timeout dla stacji 11195. Próba 1/3
14/16 | WARMIŃSKO-MAZURSKIE | Elbląg | stacja 861
15/16 | WIELKOPOLSKIE | Poznań | stacja 944
16/16 | ZACHODNIOPOMORSKIE | Szczecin | stacja 986


In [6]:
older_weather = pd.DataFrame(older_weather_records)

older_weather.shape

(280704, 10)

In [7]:
older_weather["timestamp_utc"] = pd.to_datetime(
    older_weather["timestamp_utc"],
    utc=True,
)

print(
    older_weather["timestamp_utc"].min(),
    "→",
    older_weather["timestamp_utc"].max(),
)

2023-08-20 00:00:00+00:00 → 2025-08-19 23:00:00+00:00


In [8]:
print("Nieudane stacje:", len(failed_stations))
print(failed_stations)

Nieudane stacje: 0
[]


In [9]:
older_weather.to_parquet(
    "../data/raw/open_meteo_weather_older.parquet",
    index=False,
)

In [10]:
current_weather["timestamp_utc"] = pd.to_datetime(
    current_weather["timestamp_utc"],
    utc=True,
)

In [11]:
weather_3y = pd.concat(
    [
        older_weather,
        current_weather,
    ],
    ignore_index=True,
)

In [12]:
before = len(weather_3y)

weather_3y = (
    weather_3y
    .drop_duplicates(
        subset=[
            "station_id",
            "timestamp_utc",
        ]
    )
    .reset_index(drop=True)
)

after = len(weather_3y)

print("Przed:", before)
print("Po:", after)
print("Duplikaty usunięte:", before - after)

Przed: 421248
Po: 421248
Duplikaty usunięte: 0


In [13]:
print(
    "Zakres:",
    weather_3y["timestamp_utc"].min(),
    "→",
    weather_3y["timestamp_utc"].max(),
)

print("Rekordów:", len(weather_3y))
print("Stacji:", weather_3y["station_id"].nunique())

Zakres: 2023-08-20 00:00:00+00:00 → 2026-08-20 23:00:00+00:00
Rekordów: 421248
Stacji: 16


In [14]:
weather_3y.to_parquet(
    "../data/raw/open_meteo_weather_3y.parquet",
    index=False,
)